### 1. API로 행정동단위 생활인구 받아오기 : 최근 2개월 자료만 제공하여 한계가 있음

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# .env 파일 로드 (SEOUL_API_KEY 포함)
load_dotenv()
API_KEY = os.getenv("SEOUL_API_KEY")

SERVICE = "Spop250mLocalResdDong"  # [내국인] 행정동별 서울 생활인구(250m), infId=OA-23016

# Sheet/OpenAPI는 최근 2개월 자료만 제공 -> 1~1000건 요청 시 최신 관측값을 받아옴     
url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/{SERVICE}/1/1000/"
response = requests.get(url)
response.raise_for_status()
result = response.json()[SERVICE]

print(result['RESULT'])
print('총 데이터 건수:', result['list_total_count'])

df_local_resd_dong = pd.DataFrame(result['row'])
df_local_resd_dong

{'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다'}
총 데이터 건수: 1270752


,YMD,TT,H_DNG_CD,SPOP,M00,M10,M15,M20,M25,M30,...,F25,F30,F35,F40,F45,F50,F55,F60,F65,F70
0,20260901,00,11110515,13370.17,45.55,306.13,344.80,306.78,554.13,628.60,...,535.92,521.21,445.69,529.92,617.40,626.43,575.69,533.63,459.37,1219.59
1,20260901,00,11110530,12519.14,7.59,147.60,156.33,207.68,448.39,597.21,...,475.89,609.32,594.12,572.62,549.40,582.61,565.29,445.45,453.97,1106.93
2,20260901,00,11110540,2776.42,*,29.12,293.45,185.52,119.09,93.32,...,69.00,93.79,81.56,87.14,88.04,99.61,80.01,91.16,120.30,280.86
3,20260901,00,11110550,10851.23,40.48,229.65,267.95,311.54,326.79,358.73,...,339.63,374.69,343.45,408.90,502.24,500.16,526.10,473.45,442.65,1029.41
4,20260901,00,11110560,19092.89,40.15,372.69,350.87,363.26,457.75,556.64,...,555.99,619.90,585.04,703.57,807.09,905.62,1047.52,973.89,913.27,2021.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,20260901,02,11305635,25168.25,46.60,166.79,251.73,734.16,1255.84,1310.38,...,1231.31,1067.11,655.64,680.86,724.96,949.04,1185.09,1035.07,1219.83,2381.52
996,20260901,02,11305645,22139.48,97.40,372.53,363.56,433.46,578.22,718.94,...,634.57,684.72,557.40,651.48,663.05,898.88,1143.74,1002.84,1159.98,2710.46
997,20260901,02,11305660,30308.33,91.75,415.81,572.28,596.53,909.40,1102.33,...,1058.41,977.05,721.20,849.88,974.75,1202.88,1479.68,1462.24,1615.52,3728.71
998,20260901,02,11320511,24128.03,81.11,413.07,428.45,483.09,751.91,932.23,...,841.83,896.78,757.40,815.30,962.16,1070.79,1283.52,1073.91,1201.17,2441.02


### 2. 직접 zip파일을 불러와서 DataFrame 만들기

In [2]:
import io
import zipfile
import requests
import pandas as pd

# 데이터셋(infId)마다 서버가 실제로 요구하는 seq 값 규칙이 다름 (브라우저 Network 탭으로 확인한 값)
# - OA-22850 (250_ORGN_CT, 대도시권 생활인구 원본): seq = 기준월 뒤 4자리 (yymm)
# - OA-23016 (250_LOCAL_RESD_ADMDONG, 거주인구-행정동 매핑): seq = 기준월 전체 (yyyymm)
SEQ_RULES = {
    "OA-22850": lambda period: period[2:],
    "OA-23016": lambda period: period,
}

def download_seoul_bigdata_df(period, inf_id="OA-22850", seq=None):
    """
    서울 열린데이터광장 '파일내려받기'의 원본 zip을 내려받아
    디스크에 저장하지 않고 바로 DataFrame으로 변환
    zip 안에 csv가 여러 개 들어 있으면(예: 월별 zip 안에 일별 csv 다수) 모두 읽어 하나로 concat

    period: 기준 일자/월, 'yyyymmdd' 또는 'yyyymm' 형식 문자열
    seq: 서버에 보낼 seq 값을 직접 지정. None이면 SEQ_RULES에서 inf_id에 맞는 규칙을 사용
         (모르는 infId라면 브라우저 개발자도구 Network 탭에서 실제 seq 값을 확인해 직접 지정)
    """
    if seq is None:
        if inf_id not in SEQ_RULES:
            raise ValueError(
                f"infId={inf_id}에 대한 seq 규칙을 모릅니다. "
                "브라우저 개발자도구(Network 탭)에서 실제 seq 값을 확인해 seq= 인자로 직접 지정하세요."
            )
        seq = SEQ_RULES[inf_id](period)

    url = "https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?&useCache=false"
    payload = {
        "infId": inf_id,
        "seq": seq,
        "infSeq": "1",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": f"https://data.seoul.go.kr/dataList/{inf_id}/S/1/datasetView.do",
    }

    response = requests.post(url, data=payload, headers=headers)
    response.raise_for_status()

    # zip이 아닌 응답(에러 페이지 등)이 오는 경우를 대비해 매직바이트로 사전 검증
    if not response.content.startswith(b"PK"):
        raise ValueError(
            f"zip 파일이 아닌 응답을 받았습니다 (infId={inf_id}, period={period}, seq={seq}).\n"
            "seq 값이 잘못됐거나 해당 연월의 데이터가 아직 없을 수 있습니다.\n"
            f"응답 내용(일부): {response.content[:500]!r}"
        )

    # zip 파일을 메모리 상에서 바로 열어 안의 csv를 전부 읽어 concat
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        csv_names = sorted(name for name in zf.namelist() if name.lower().endswith(".csv"))
        df_parts = []
        for name in csv_names:
            with zf.open(name) as f:
                df_parts.append(pd.read_csv(f, encoding="cp949"))
        df = pd.concat(df_parts, ignore_index=True) if len(df_parts) > 1 else df_parts[0]

    print(f"{len(csv_names)}개 csv 불러오기 완료 ({len(df):,} rows): {', '.join(csv_names)}")
    return df

In [3]:
# 250_LOCAL_RESD_ADMDONG_202607.zip 다운로드 -> DataFrame 변환 (zip 안 일별 csv 31개 자동 concat)
df_local_resd_admdong = download_seoul_bigdata_df("202607", inf_id="OA-23016")
df_local_resd_admdong

31개 csv 불러오기 완료 (317,688 rows): 250_LOCAL_RESD_ADMDONG_20260701.csv, 250_LOCAL_RESD_ADMDONG_20260702.csv, 250_LOCAL_RESD_ADMDONG_20260703.csv, 250_LOCAL_RESD_ADMDONG_20260704.csv, 250_LOCAL_RESD_ADMDONG_20260705.csv, 250_LOCAL_RESD_ADMDONG_20260706.csv, 250_LOCAL_RESD_ADMDONG_20260707.csv, 250_LOCAL_RESD_ADMDONG_20260708.csv, 250_LOCAL_RESD_ADMDONG_20260709.csv, 250_LOCAL_RESD_ADMDONG_20260710.csv, 250_LOCAL_RESD_ADMDONG_20260711.csv, 250_LOCAL_RESD_ADMDONG_20260712.csv, 250_LOCAL_RESD_ADMDONG_20260713.csv, 250_LOCAL_RESD_ADMDONG_20260714.csv, 250_LOCAL_RESD_ADMDONG_20260715.csv, 250_LOCAL_RESD_ADMDONG_20260716.csv, 250_LOCAL_RESD_ADMDONG_20260717.csv, 250_LOCAL_RESD_ADMDONG_20260718.csv, 250_LOCAL_RESD_ADMDONG_20260719.csv, 250_LOCAL_RESD_ADMDONG_20260720.csv, 250_LOCAL_RESD_ADMDONG_20260721.csv, 250_LOCAL_RESD_ADMDONG_20260722.csv, 250_LOCAL_RESD_ADMDONG_20260723.csv, 250_LOCAL_RESD_ADMDONG_20260724.csv, 250_LOCAL_RESD_ADMDONG_20260725.csv, 250_LOCAL_RESD_ADMDONG_20260726.csv, 250_LO

,일자,시간,행정동코드,생활인구합계,남자 0~9세,남자 10~14세,남자 15~19세,남자 20~24세,남자 25~29세,남자 30~34세,...,여자 25~29세,여자 30~34세,여자 35~39세,여자 40~44세,여자 45~49세,여자 50~54세,여자 55~59세,여자 60~64세,여자 65~69세,여자 70세 이상
0,20260701,0.0,11110515.0,13232.33,69.82,286.59,357.85,312.76,546.42,524.37,...,452.22,542.16,480.98,548.62,623.64,604.74,567.76,497.39,451.06,1228.44
1,20260701,0.0,11110530.0,11927.14,12.54,139.95,124.37,211.63,465.27,592.49,...,516.06,599.46,569.66,523.61,590.67,488.74,540.74,386.70,414.19,952.77
2,20260701,0.0,11110540.0,2751.88,*,51.14,224.97,140.61,126.23,88.41,...,85.37,72.06,93.30,99.37,81.11,100.93,80.89,107.08,111.78,296.23
3,20260701,0.0,11110550.0,10646.53,50.53,215.78,296.34,280.11,339.20,375.15,...,362.17,348.90,339.86,445.56,475.84,493.97,535.79,482.49,450.19,955.18
4,20260701,0.0,11110560.0,18809.07,40.72,394.23,361.64,395.30,506.14,541.60,...,570.30,640.21,573.77,745.61,819.87,911.53,1043.41,911.67,868.14,1887.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317683,"20260731;""23"";""11740650 "";""26939.06"";""80.0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
317684,"20260731;""23"";""11740660 "";""28822.06"";""78.7...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
317685,"20260731;""23"";""11740685 "";""59830.39"";""259....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
317686,"20260731;""23"";""11740690 "";""35108.00"";""224....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# 250_LOCAL_RESD_ADMDONG_202501.zip ~ 250_LOCAL_RESD_ADMDONG_202512.zip (2025년 12개월치)를
# 불러와 하나의 DataFrame으로 stack (각 월 zip 안의 일별 csv는 download_seoul_bigdata_df가 자동으로 concat)

months = pd.date_range("2025-01-01", "2025-12-01", freq="MS").strftime("%Y%m").tolist()

df_list = []
for ym in months:
    df_month = download_seoul_bigdata_df(ym, inf_id="OA-23016")
    df_list.append(df_month)

df_local_resd_admdong_1y = pd.concat(df_list, ignore_index=True)
print(f"전체 {len(df_local_resd_admdong_1y):,} rows ({len(months)}개월 stack 완료)")
df_local_resd_admdong_1y

C:\Users\leeja\AppData\Local\Temp\ipykernel_24812\857090416.py:60: DtypeWarning: Columns (0: 남자 10~14세, 1: 여자 10~14세, 2: 여자 15~19세) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parts.append(pd.read_csv(f, encoding="cp949"))


1개 csv 불러오기 완료 (316,944 rows): 250_LOCAL_RESD_ADMDONG_202501.csv


C:\Users\leeja\AppData\Local\Temp\ipykernel_24812\857090416.py:60: DtypeWarning: Columns (0: 남자 10~14세, 1: 여자 15~19세) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parts.append(pd.read_csv(f, encoding="cp949"))


1개 csv 불러오기 완료 (286,272 rows): 250_LOCAL_RESD_ADMDONG_202502.csv


C:\Users\leeja\AppData\Local\Temp\ipykernel_24812\857090416.py:60: DtypeWarning: Columns (0: 남자 10~14세, 1: 남자 15~19세, 2: 여자 10~14세, 3: 여자 15~19세) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parts.append(pd.read_csv(f, encoding="cp949"))


...

C:\Users\leeja\AppData\Local\Temp\ipykernel_24812\857090416.py:60: DtypeWarning: Columns (0: 남자 10~14세, 1: 남자 15~19세, 2: 남자 20~24세, 3: 남자 25~29세, 4: 남자 30~34세, 5: 남자 35~39세, 6: 남자 40~44세, 7: 남자 45~49세, 8: 남자 50~54세, 9: 남자 55~59세, 10: 남자 60~64세, 11: 남자 65~69세, 12: 남자 70세 이상, 13: 여자 10~14세, 14: 여자 15~19세, 15: 여자 20~24세, 16: 여자 25~29세, 17: 여자 30~34세, 18: 여자 35~39세, 19: 여자 40~44세, 20: 여자 45~49세, 21: 여자 50~54세, 22: 여자 55~59세, 23: 여자 60~64세, 24: 여자 65~69세, 25: 여자 70세 이상) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parts.append(pd.read_csv(f, encoding="cp949"))


1개 csv 불러오기 완료 (307,820 rows): 250_LOCAL_RESD_ADMDONG_202509.csv


C:\Users\leeja\AppData\Local\Temp\ipykernel_24812\857090416.py:60: DtypeWarning: Columns (0: 남자 10~14세, 1: 남자 15~19세, 2: 여자 0~9세, 3: 여자 10~14세, 4: 여자 15~19세) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parts.append(pd.read_csv(f, encoding="cp949"))


1개 csv 불러오기 완료 (317,688 rows): 250_LOCAL_RESD_ADMDONG_202510.csv
1개 csv 불러오기 완료 (307,440 rows): 250_LOCAL_RESD_ADMDONG_202511.csv
1개 csv 불러오기 완료 (317,688 rows): 250_LOCAL_RESD_ADMDONG_202512.csv
전체 3,736,080 rows (12개월 stack 완료)


,일자,시간,행정동코드,생활인구합계,남자 0~9세,남자 10~14세,남자 15~19세,남자 20~24세,남자 25~29세,남자 30~34세,...,여자 25~29세,여자 30~34세,여자 35~39세,여자 40~44세,여자 45~49세,여자 50~54세,여자 55~59세,여자 60~64세,여자 65~69세,여자 70세 이상
0,20250101,0,11110515,14205.95,70.38,334.38,334.66,367.19,484.77,556.61,...,593.46,581.71,511.42,653.24,630.62,643.54,670.5,528.11,521.26,1214.13
1,20250101,0,11110530,15529.34,29.76,163.7,319.78,517.28,674.24,792.99,...,803.64,832.55,563.35,638.31,651.6,688.84,672.42,624.14,491.67,1122.42
2,20250101,0,11110540,3296.03,6.22,43.35,71.11,186.44,112.87,126.32,...,152.82,121.76,120.98,113.75,95.02,127.55,134.63,152.65,134.0,323.15
3,20250101,0,11110550,11377.86,70.98,254.08,257.4,307.75,347.25,384.48,...,353.23,446.27,389.03,442.82,485.53,602.53,554.7,468.82,516.46,1033.39
4,20250101,0,11110560,19725.03,108.25,371.79,417.22,534.38,595.75,670.16,...,651.68,648.71,670.66,830.18,756.98,960.8,1124.71,1023.73,836.05,1606.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3736075,20251231,23,11740650,28055.36,120.65,356.53,412.26,552.5,1199.52,1737.74,...,1454.36,1634.86,995.73,955.11,925.13,980.26,1067.87,1153.46,1188.68,2395.96
3736076,20251231,23,11740660,32534.43,132.33,546.7,742.54,783.41,1279.3,1654.65,...,1351.06,1428.8,1062.3,1192.47,1426.01,1333.97,1364.64,1378.59,1395.76,2552.97
3736077,20251231,23,11740685,57760.72,245.86,774.12,919.83,1204.04,2242.3,2874.29,...,2538.07,2629.96,2047.19,2081.81,2008.07,2422.56,2411.42,2477.94,2633.85,4472.0
3736078,20251231,23,11740690,32538.78,296.31,1053.99,742.98,553.71,679.56,1263.12,...,762.69,1518.2,1682.73,1967.59,1566.98,1340.92,1290.16,1364.51,1295.47,1614.57


In [5]:
df_local_resd_admdong_1y.tail(10)

,일자,시간,행정동코드,생활인구합계,남자 0~9세,남자 10~14세,남자 15~19세,남자 20~24세,남자 25~29세,남자 30~34세,...,여자 25~29세,여자 30~34세,여자 35~39세,여자 40~44세,여자 45~49세,여자 50~54세,여자 55~59세,여자 60~64세,여자 65~69세,여자 70세 이상
3736070,20251231,23,11740590,23084.00,192.69,706.12,546.32,406.73,556.1,720.46,...,642.81,739.91,895.68,1257.74,1182.71,1018.47,1089.38,1004.68,831.91,1698.61
3736071,20251231,23,11740600,25794.92,121.89,345.27,336.49,518.71,967.5,1229.63,...,852.03,955.25,807.46,874.99,838.0,964.4,1120.91,1227.46,1321.8,2626.75
3736072,20251231,23,11740610,41577.82,128.88,507.58,665.32,827.71,1695.23,2238.73,...,1764.07,2087.25,1575.3,1580.63,1561.56,1755.78,1723.96,1799.13,1749.29,3436.32
3736073,20251231,23,11740620,33196.20,98.14,301.35,549.2,983.07,1947.92,2171.17,...,1998.13,1760.56,1199.39,1075.3,1022.29,1165.98,1165.4,1257.38,1300.8,2505.34
3736074,20251231,23,11740640,20217.43,78.77,453.18,363.75,367.16,595.38,783.75,...,699.37,845.59,750.32,855.29,866.26,901.36,992.55,860.16,873.85,1621.8
3736075,20251231,23,11740650,28055.36,120.65,356.53,412.26,552.5,1199.52,1737.74,...,1454.36,1634.86,995.73,955.11,925.13,980.26,1067.87,1153.46,1188.68,2395.96
3736076,20251231,23,11740660,32534.43,132.33,546.7,742.54,783.41,1279.3,1654.65,...,1351.06,1428.8,1062.3,1192.47,1426.01,1333.97,1364.64,1378.59,1395.76,2552.97
3736077,20251231,23,11740685,57760.72,245.86,774.12,919.83,1204.04,2242.3,2874.29,...,2538.07,2629.96,2047.19,2081.81,2008.07,2422.56,2411.42,2477.94,2633.85,4472.0
3736078,20251231,23,11740690,32538.78,296.31,1053.99,742.98,553.71,679.56,1263.12,...,762.69,1518.2,1682.73,1967.59,1566.98,1340.92,1290.16,1364.51,1295.47,1614.57
3736079,20251231,23,11740700,29629.31,189.55,647.02,564.17,565.81,915.46,1204.66,...,1026.63,1214.76,1139.45,1073.25,1117.27,1175.77,1470.75,1345.77,1391.69,2411.62


In [6]:
df_2025 = df_local_resd_admdong_1y.copy()

In [7]:
# 컬럼명 변경
df_2025.columns = ['YMD', 'TT', 'H_DNG_CD', 'SPOP', 
                   'M0009','M1014','M1519','M2024','M2529','M3034','M3539',
                   'M4044','M4549','M5054','M5559','M6064','M6569','M7000',
                   'F0009','F1014','F1519','F2024','F2529','F3034','F3539',
                   'F4044','F4549','F5054','F5559','F6064','F6569','F7000'
                   ]

df_2025

,YMD,TT,H_DNG_CD,SPOP,M0009,M1014,M1519,M2024,M2529,M3034,...,F2529,F3034,F3539,F4044,F4549,F5054,F5559,F6064,F6569,F7000
0,20250101,0,11110515,14205.95,70.38,334.38,334.66,367.19,484.77,556.61,...,593.46,581.71,511.42,653.24,630.62,643.54,670.5,528.11,521.26,1214.13
1,20250101,0,11110530,15529.34,29.76,163.7,319.78,517.28,674.24,792.99,...,803.64,832.55,563.35,638.31,651.6,688.84,672.42,624.14,491.67,1122.42
2,20250101,0,11110540,3296.03,6.22,43.35,71.11,186.44,112.87,126.32,...,152.82,121.76,120.98,113.75,95.02,127.55,134.63,152.65,134.0,323.15
3,20250101,0,11110550,11377.86,70.98,254.08,257.4,307.75,347.25,384.48,...,353.23,446.27,389.03,442.82,485.53,602.53,554.7,468.82,516.46,1033.39
4,20250101,0,11110560,19725.03,108.25,371.79,417.22,534.38,595.75,670.16,...,651.68,648.71,670.66,830.18,756.98,960.8,1124.71,1023.73,836.05,1606.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3736075,20251231,23,11740650,28055.36,120.65,356.53,412.26,552.5,1199.52,1737.74,...,1454.36,1634.86,995.73,955.11,925.13,980.26,1067.87,1153.46,1188.68,2395.96
3736076,20251231,23,11740660,32534.43,132.33,546.7,742.54,783.41,1279.3,1654.65,...,1351.06,1428.8,1062.3,1192.47,1426.01,1333.97,1364.64,1378.59,1395.76,2552.97
3736077,20251231,23,11740685,57760.72,245.86,774.12,919.83,1204.04,2242.3,2874.29,...,2538.07,2629.96,2047.19,2081.81,2008.07,2422.56,2411.42,2477.94,2633.85,4472.0
3736078,20251231,23,11740690,32538.78,296.31,1053.99,742.98,553.71,679.56,1263.12,...,762.69,1518.2,1682.73,1967.59,1566.98,1340.92,1290.16,1364.51,1295.47,1614.57


In [8]:
# 연령대별 생활인구에 3명 이하는 "*"로 비식별화 처리가 되어 있음 
df_2025[df_2025['M0009'] == '*']

,YMD,TT,H_DNG_CD,SPOP,M0009,M1014,M1519,M2024,M2529,M3034,...,F2529,F3034,F3539,F4044,F4549,F5054,F5559,F6064,F6569,F7000
22,20250101,0,11140590,12849.40,*,74.01,297.03,715.69,1043.14,1016.39,...,1117.03,759.12,459.16,342.51,339.65,315.63,331.32,294.21,258.91,463.74
232,20250101,0,11470610,6989.45,*,117.36,136.7,111.59,210.74,236.41,...,164.15,216.98,212.44,268.38,281.7,291.5,386.87,376.96,383.32,537.56
448,20250101,1,11140590,11333.37,*,64.79,230.03,600.24,923.05,881.39,...,934.67,678.93,402.51,309.02,291.7,284.61,309.71,273.43,237.75,487.23
658,20250101,1,11470610,7016.39,*,112.28,145.05,129.27,193.54,236.61,...,159.97,216.38,226.52,276.34,291.11,291.47,377.01,371.0,379.07,565.43
854,20250101,2,11110540,3001.41,*,43.31,47.24,169.53,97.34,124.4,...,117.22,100.21,116.17,117.47,94.92,117.99,115.55,129.12,125.02,324.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3734819,20251231,21,11140570,11801.70,*,108.39,144.79,470.37,691.78,701.92,...,743.69,565.7,361.51,363.9,353.88,462.76,455.49,321.94,413.49,660.81
3734862,20251231,21,11200720,10801.67,*,130.49,111.98,278.01,511.11,593.92,...,484.24,501.77,330.89,365.63,358.45,458.48,467.62,497.56,437.83,726.2
3735289,20251231,22,11200720,11084.19,*,125.61,151.95,288.32,523.34,605.93,...,488.31,518.89,331.58,414.98,362.26,464.45,469.33,536.76,447.46,729.26
3735676,20251231,23,11140605,9877.10,*,52.93,82.79,609.35,1062.69,891.33,...,1175.81,642.52,289.43,228.63,257.09,229.82,197.92,124.57,105.91,168.32


In [9]:
# 연령대별 생활인구에 3명 이하는 "*"로 비식별화 처리가 되어 있어, str또는 object로 읽어야 함
df_2025.dtypes

YMD           int64
TT            int64
H_DNG_CD      int64
SPOP        float64
M0009           str
M1014        object
M1519        object
M2024        object
M2529        object
M3034        object
M3539        object
M4044        object
M4549        object
M5054        object
M5559        object
M6064        object
M6569        object
M7000        object
F0009        object
F1014        object
F1519        object
F2024        object
F2529        object
F3034        object
F3539        object
F4044        object
F4549        object
F5054        object
F5559        object
F6064        object
F6569        object
F7000        object
dtype: object

In [10]:
import re

# 비식별 처리된 "*"(3명 이하)를 1.5로 대체하고 숫자형으로 변환
age_cols = [c for c in df_2025.columns if re.fullmatch(r'[MF]\d{4}', c)]

df_2025[age_cols] = (
    df_2025[age_cols]
    .replace('*', 1.5)
    .astype(float)
)

df_2025.dtypes

YMD           int64
TT            int64
H_DNG_CD      int64
SPOP        float64
M0009       float64
M1014       float64
M1519       float64
M2024       float64
M2529       float64
M3034       float64
M3539       float64
M4044       float64
M4549       float64
M5054       float64
M5559       float64
M6064       float64
M6569       float64
M7000       float64
F0009       float64
F1014       float64
F1519       float64
F2024       float64
F2529       float64
F3034       float64
F3539       float64
F4044       float64
F4549       float64
F5054       float64
F5559       float64
F6064       float64
F6569       float64
F7000       float64
dtype: object

In [11]:
# 모든 컬럼의 결측치 개수 확인 
df_2025.count()

YMD         3736080
TT          3736080
H_DNG_CD    3736080
SPOP        3736080
M0009       3736080
M1014       3736080
M1519       3736080
M2024       3736080
M2529       3736080
M3034       3736080
M3539       3736080
M4044       3736080
M4549       3736080
M5054       3736080
M5559       3736080
M6064       3736080
M6569       3736080
M7000       3736080
F0009       3736080
F1014       3736080
F1519       3736080
F2024       3736080
F2529       3736080
F3034       3736080
F3539       3736080
F4044       3736080
F4549       3736080
F5054       3736080
F5559       3736080
F6064       3736080
F6569       3736080
F7000       3736080
dtype: int64

In [12]:
AGE_BINS = {
    '00': ['0009'],
    '10': ['1014', '1519'],
    '20': ['2024', '2529'],
    '30': ['3034', '3539'],
    '40': ['4044', '4549'],
    '50': ['5054', '5559'],
    '60': ['6064', '6569'],
    '70': ['7000'],
}

def living_10(df):
    for decade, suffixes in AGE_BINS.items():
        M_cols = [f'M{s}' for s in suffixes]
        F_cols = [f'F{s}' for s in suffixes]
        df[f'M{decade}'] = df[M_cols].sum(axis=1).round(2)
        df[f'F{decade}'] = df[F_cols].sum(axis=1).round(2)

    cols = ['YMD', 'TT', 'H_DNG_CD', 'SPOP']
    cols += [f'M{d}' for d in AGE_BINS] + [f'F{d}' for d in AGE_BINS]
    return df[cols]


# 함수 living_10을 이용해 10세 단위 연령대별 생활인구를 계산 
SPOP_2025 = living_10(df_2025)
SPOP_2025

,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,M60,M70,F00,F10,F20,F30,F40,F50,F60,F70
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,904.16,896.48,93.53,661.81,918.09,1093.13,1283.86,1314.04,1049.37,1214.13
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,978.95,782.59,40.81,511.21,1402.52,1395.90,1289.91,1361.26,1115.81,1122.42
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,213.26,257.14,31.19,91.79,229.92,242.74,208.77,262.18,286.65,323.15
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,854.41,781.00,70.81,386.17,588.42,835.30,928.35,1157.23,985.28,1033.39
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,1697.43,1402.08,67.69,693.08,1123.64,1319.37,1587.16,2085.51,1859.78,1606.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3736075,20251231,23,11740650,28055.36,120.65,768.79,1752.02,2940.46,2088.57,2140.90,2126.76,1845.94,133.15,674.72,2166.34,2630.59,1880.24,2048.13,2342.14,2395.96
3736076,20251231,23,11740660,32534.43,132.33,1289.24,2062.71,2878.61,2547.59,2678.11,2432.25,1961.14,110.87,1163.30,2142.77,2491.10,2618.48,2698.61,2774.35,2552.97
3736077,20251231,23,11740685,57760.72,245.86,1693.95,3446.34,5226.51,4626.54,4913.62,4842.98,3828.14,142.58,1658.42,3950.98,4677.15,4089.88,4833.98,5111.79,4472.00
3736078,20251231,23,11740690,32538.78,296.31,1796.97,1233.27,2724.20,3389.36,2383.87,2334.93,1565.98,283.61,1516.95,1372.20,3200.93,3534.57,2631.08,2659.98,1614.57


In [13]:
# 행정동 수 확인
len(SPOP_2025.H_DNG_CD.unique())

428

In [14]:
# 일 수 확인
len(SPOP_2025.YMD.unique())

365

In [15]:
# 시간 수 확인
len(SPOP_2025.TT.unique())

24

In [16]:
# 행정동 수도 너무 많고, 기대되는 관찰값 갯수와 데이터에 포함된 관찰값 갯수를 비교해보면, 데이터에 결측치가 많음을 알 수 있음

428 * 365 * 24

3749280

### 3. 행정동 코드정보 조인

- https://data.seoul.go.kr/dataVisual/seoul/seoulLivingPopulation.do 페이지의 "데이터 활용 가이드 > 전국 행정동 코드정보"
- 이 링크는 데이터셋별 infId(OA-XXXXX)가 아니라 `infId=DOWNLOAD`, `infSeq=4`, `seq=18` 고정값으로 같은 `nio_download.do` 엔드포인트에 POST함 (페이지 소스의 `downloadFile(4,'18')` 확인)
- zip 안에 2023.01~현재까지 월별 스냅샷(`ADMI_YYYYMM.csv`)이 들어 있으며, 인코딩은 다른 생활인구 파일과 달리 UTF-8(BOM)

In [17]:
import io
import zipfile
import requests
import pandas as pd

response = requests.post(
    "https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?&useCache=false",
    data={"infId": "DOWNLOAD", "infSeq": "4", "seq": "18"},
    headers={
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://data.seoul.go.kr/dataVisual/seoul/seoulLivingPopulation.do",
    },
)
response.raise_for_status()

months = [f"2025{m:02d}" for m in range(1, 13)]  # 202501~202512

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    df_parts = []
    for ym in months:
        with zf.open(f"SEOUL_ADMI/ADMI_{ym}.csv") as f:
            df_parts.append(pd.read_csv(f, encoding="utf-8-sig"))
    df_admdong_code = pd.concat(df_parts, ignore_index=True)

print(f"{len(months)}개월치 불러오기 완료 ({len(df_admdong_code):,} rows)")
df_admdong_code

12개월치 불러오기 완료 (42,719 rows)


,SIDO_NM,SGG_NM,ADMI_NM,ADMI_CD,FULL_NM,BASE_YM
0,서울특별시,종로구,청운효자동,11110515,서울특별시 종로구 청운효자동,202501
1,서울특별시,종로구,사직동,11110530,서울특별시 종로구 사직동,202501
2,서울특별시,종로구,삼청동,11110540,서울특별시 종로구 삼청동,202501
3,서울특별시,종로구,부암동,11110550,서울특별시 종로구 부암동,202501
4,서울특별시,종로구,평창동,11110560,서울특별시 종로구 평창동,202501
...,...,...,...,...,...,...
42714,전북특별자치도,부안군,백산면,52800380,전북특별자치도 부안군 백산면,202512
42715,전북특별자치도,부안군,상서면,52800390,전북특별자치도 부안군 상서면,202512
42716,전북특별자치도,부안군,하서면,52800400,전북특별자치도 부안군 하서면,202512
42717,전북특별자치도,부안군,줄포면,52800410,전북특별자치도 부안군 줄포면,202512


In [18]:
df_admdong_code.dtypes

SIDO_NM      str
SGG_NM       str
ADMI_NM      str
ADMI_CD    int64
FULL_NM      str
BASE_YM    int64
dtype: object

In [19]:
# 서울시 행정동 개수가 월별로 실제 몇 개였는지 확인 (연중 분동/통합으로 개수가 바뀜)
seoul_monthly_dong_count = (
    df_admdong_code[df_admdong_code['SIDO_NM'] == '서울특별시']
    .groupby('BASE_YM')['ADMI_CD']
    .nunique()
)
seoul_monthly_dong_count

BASE_YM
202501    426
202502    426
202503    426
202504    426
202505    426
202506    426
202507    426
202508    427
202509    427
202510    427
202511    427
202512    427
Name: ADMI_CD, dtype: int64

In [20]:
# 202507 -> 202508 사이에 어떤 행정동이 생기고 없어졌는지 확인
seoul = df_admdong_code[df_admdong_code['SIDO_NM'] == '서울특별시'].copy()
seoul['BASE_YM'] = seoul['BASE_YM'].astype(str)

jul_codes = set(seoul.loc[seoul['BASE_YM'] == '202507', 'ADMI_CD'])
aug_codes = set(seoul.loc[seoul['BASE_YM'] == '202508', 'ADMI_CD'])

changed_codes = (jul_codes - aug_codes) | (aug_codes - jul_codes)
seoul[(seoul['BASE_YM'].isin(['202507', '202508'])) & (seoul['ADMI_CD'].isin(changed_codes))]

,SIDO_NM,SGG_NM,ADMI_NM,ADMI_CD,FULL_NM,BASE_YM
21436,서울특별시,동대문구,용신동,11230536,서울특별시 동대문구 용신동,202507
24995,서울특별시,동대문구,신설동,11230515,서울특별시 동대문구 신설동,202508
24996,서울특별시,동대문구,용두동,11230533,서울특별시 동대문구 용두동,202508


In [21]:
# 8월부터 등장한 신설동(11230515)·용두동(11230533)과,
# 8~9월에 일부 시간대만 남아있던 구코드 용신동(11230536)을 모두 합쳐
# 하나의 용신동(11230536)으로 통일 (1~7월은 원래 용신동 코드만 있으므로 결과가 그대로 유지됨)

MERGE_CODES = [11230515, 11230533, 11230536]  # 신설동, 용두동, (구)용신동
TARGET_CODE = 11230536                         # 통일할 코드: 용신동

value_cols = [c for c in SPOP_2025.columns if c not in ('YMD', 'TT', 'H_DNG_CD')]

merged = (
    SPOP_2025[SPOP_2025['H_DNG_CD'].isin(MERGE_CODES)]
    .groupby(['YMD', 'TT'], as_index=False)[value_cols]
    .sum()
)
merged['H_DNG_CD'] = TARGET_CODE
merged = merged[SPOP_2025.columns]  # 원래 컬럼 순서 유지

SPOP_2025_426 = pd.concat(
    [SPOP_2025[~SPOP_2025['H_DNG_CD'].isin(MERGE_CODES)], merged],
    ignore_index=True
)

print(SPOP_2025['H_DNG_CD'].nunique(), '->', SPOP_2025_426['H_DNG_CD'].nunique())
SPOP_2025_426

428 -> 426


,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,M60,M70,F00,F10,F20,F30,F40,F50,F60,F70
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,904.16,896.48,93.53,661.81,918.09,1093.13,1283.86,1314.04,1049.37,1214.13
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,978.95,782.59,40.81,511.21,1402.52,1395.90,1289.91,1361.26,1115.81,1122.42
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,213.26,257.14,31.19,91.79,229.92,242.74,208.77,262.18,286.65,323.15
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,854.41,781.00,70.81,386.17,588.42,835.30,928.35,1157.23,985.28,1033.39
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,1697.43,1402.08,67.69,693.08,1123.64,1319.37,1587.16,2085.51,1859.78,1606.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731755,20251231,19,11230536,41645.33,111.51,860.29,2556.25,3803.83,3484.63,3891.25,3836.06,2515.58,119.70,815.52,2958.74,3453.88,3053.85,3833.02,3375.74,2975.48
3731756,20251231,20,11230536,41793.42,111.40,878.63,2663.29,3991.45,3503.55,3845.54,3675.41,2366.42,116.73,870.00,2973.76,3693.43,3121.59,3751.92,3308.61,2921.69
3731757,20251231,21,11230536,39425.33,105.04,821.77,2491.18,3914.05,3263.19,3570.64,3408.06,2104.43,110.66,822.31,2889.11,3584.42,3004.94,3609.39,3006.68,2719.46
3731758,20251231,22,11230536,40861.42,103.68,847.92,2661.42,4094.05,3458.72,3639.98,3399.82,2213.12,116.22,887.72,2972.74,3738.67,3119.05,3672.85,3067.54,2867.92


In [22]:
426 * 365 * 24

3731760

In [23]:
# 행정동 수 확인 : 일치함
len(SPOP_2025_426.H_DNG_CD.unique())

426

In [24]:
# df_admdong_code에서 1월(202501) 행정동 코드만 추출해 SPOP_2025_426과 조인 (426개 기준과 맞음)
admdong_202501 = df_admdong_code[
    (df_admdong_code['SIDO_NM'] == '서울특별시') &
    (df_admdong_code['BASE_YM'].astype(str) == '202501')
]

SPOP_2025_ADM = SPOP_2025_426.merge(
    admdong_202501[['ADMI_CD', 'SIDO_NM', 'SGG_NM', 'ADMI_NM', 'FULL_NM']],
    left_on='H_DNG_CD',
    right_on='ADMI_CD',
    how='left',
)

SPOP_2025_ADM

,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,F30,F40,F50,F60,F70,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,...,1093.13,1283.86,1314.04,1049.37,1214.13,11110515,서울특별시,종로구,청운효자동,서울특별시 종로구 청운효자동
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,...,1395.90,1289.91,1361.26,1115.81,1122.42,11110530,서울특별시,종로구,사직동,서울특별시 종로구 사직동
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,...,242.74,208.77,262.18,286.65,323.15,11110540,서울특별시,종로구,삼청동,서울특별시 종로구 삼청동
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,...,835.30,928.35,1157.23,985.28,1033.39,11110550,서울특별시,종로구,부암동,서울특별시 종로구 부암동
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,...,1319.37,1587.16,2085.51,1859.78,1606.73,11110560,서울특별시,종로구,평창동,서울특별시 종로구 평창동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731755,20251231,19,11230536,41645.33,111.51,860.29,2556.25,3803.83,3484.63,3891.25,...,3453.88,3053.85,3833.02,3375.74,2975.48,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동
3731756,20251231,20,11230536,41793.42,111.40,878.63,2663.29,3991.45,3503.55,3845.54,...,3693.43,3121.59,3751.92,3308.61,2921.69,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동
3731757,20251231,21,11230536,39425.33,105.04,821.77,2491.18,3914.05,3263.19,3570.64,...,3584.42,3004.94,3609.39,3006.68,2719.46,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동
3731758,20251231,22,11230536,40861.42,103.68,847.92,2661.42,4094.05,3458.72,3639.98,...,3738.67,3119.05,3672.85,3067.54,2867.92,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동


### 4. 요일, 공휴일 정보 조인

In [25]:
# 2025년 한국 공휴일 (대체공휴일·임시공휴일 포함) - YMD(YYYYMMDD int)와 바로 비교 가능한 형식
holidays_2025 = [
    20250101,  # 신정
    20250127,  # 임시공휴일 (설 연휴 연장)
    20250128,  # 설날 연휴
    20250129,  # 설날
    20250130,  # 설날 연휴
    20250301,  # 삼일절
    20250303,  # 삼일절 대체공휴일 (3/1이 토요일)
    20250505,  # 어린이날 · 부처님오신날 (겹침)
    20250506,  # 대체공휴일 (어린이날·부처님오신날 겹침)
    20250603,  # 21대 대통령선거일
    20250606,  # 현충일
    20250815,  # 광복절
    20251003,  # 개천절
    20251005,  # 추석 연휴
    20251006,  # 추석
    20251007,  # 추석 연휴
    20251008,  # 대체공휴일 (추석 연휴 첫날이 일요일과 겹침)
    20251009,  # 한글날
    20251225,  # 크리스마스
]

len(holidays_2025)

19

In [26]:
# 공휴일이면 1, 아니면 0
SPOP_2025_ADM['HDAYS'] = SPOP_2025_ADM['YMD'].isin(holidays_2025).astype(int)

SPOP_2025_ADM['HDAYS'].value_counts()

HDAYS
0    3537504
1     194256
Name: count, dtype: int64

In [27]:
# 토요일·일요일이면 1, 월~금이면 0
SPOP_2025_ADM['WEEKEND'] = (
    pd.to_datetime(SPOP_2025_ADM['YMD'], format='%Y%m%d').dt.weekday >= 5  # 0~4: 월~금, 5~6: 토·일
).astype(int)

SPOP_2025_ADM['WEEKEND'].value_counts()

WEEKEND
0    2668464
1    1063296
Name: count, dtype: int64

In [28]:
# 공휴일도 주말도 아닌 평일(근무일)이면 1, 아니면 0
SPOP_2025_ADM['WORKDAY'] = (
    (SPOP_2025_ADM['HDAYS'] == 0) & (SPOP_2025_ADM['WEEKEND'] == 0)
).astype(int)

SPOP_2025_ADM['WORKDAY'].value_counts()

WORKDAY
1    2494656
0    1237104
Name: count, dtype: int64

In [29]:
SPOP_2025_ADM

,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,F60,F70,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,HDAYS,WEEKEND,WORKDAY
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,...,1049.37,1214.13,11110515,서울특별시,종로구,청운효자동,서울특별시 종로구 청운효자동,1,0,0
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,...,1115.81,1122.42,11110530,서울특별시,종로구,사직동,서울특별시 종로구 사직동,1,0,0
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,...,286.65,323.15,11110540,서울특별시,종로구,삼청동,서울특별시 종로구 삼청동,1,0,0
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,...,985.28,1033.39,11110550,서울특별시,종로구,부암동,서울특별시 종로구 부암동,1,0,0
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,...,1859.78,1606.73,11110560,서울특별시,종로구,평창동,서울특별시 종로구 평창동,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731755,20251231,19,11230536,41645.33,111.51,860.29,2556.25,3803.83,3484.63,3891.25,...,3375.74,2975.48,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731756,20251231,20,11230536,41793.42,111.40,878.63,2663.29,3991.45,3503.55,3845.54,...,3308.61,2921.69,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731757,20251231,21,11230536,39425.33,105.04,821.77,2491.18,3914.05,3263.19,3570.64,...,3006.68,2719.46,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731758,20251231,22,11230536,40861.42,103.68,847.92,2661.42,4094.05,3458.72,3639.98,...,3067.54,2867.92,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1


In [30]:
SPOP_2025_ADM.to_csv('data/SPOP_2025_ADM.csv', index=False, encoding='utf-8-sig')